In [ ]:
#Ej 1

# Genera todos los estados posibles (nodo, dinero)
estados = []
for nodo in range(4):
    for dinero in range(11):
        estados.append((nodo, dinero))

# Grafo de vecinos para cada nodo
vecinos = {
    0: [1,2],
    1: [0,3],
    2: [0,3],
    3: []
}

def step(estado,accion):
    """
    Dada una acción, devuelve el siguiente estado y la recompensa.
    """
    nodo, dinero = estado
    nodo_sig = accion
    dinero_sig = dinero
    recompensa = 0

    if nodo_sig == 1 and dinero >=5:
        dinero_sig = max(0, dinero - 5)
        recompensa += 3
    elif nodo_sig == 2:
        if dinero_sig < 10:
            dinero_sig += 1
        recompensa -= 1
    elif nodo_sig == 3:
        if dinero_sig > 4:
            recompensa += 2
        else:
            return (None, -2000)
    elif dinero_sig == 0:
        return (None, -2000)

    estado_sig = (nodo_sig, dinero_sig)
    return (estado_sig, recompensa)

# Construye diccionario de transiciones para cada estado-acción
transiciones = {}
for s in estados:
    transiciones[s] = {}
    nodo, _ = s
    for v in vecinos[nodo]:
        s_sig, r = step(s, v)
        transiciones[s][v] = (s_sig, r)

def value_iteration(gamma, tol=0.1, max_iters=1000):
    """
    Ejecuta iteración de valor y extrae la política óptima.
    """
    V = {s: 0.0 for s in estados}
    for _ in range(max_iters):
        delta = 0
        for s in estados:
            if s is None:
                continue
            # Calcula Q(s,a) para cada acción
            q_vals = []
            for a, (s_next, r) in transiciones[s].items():
                q_vals.append(r + gamma * (V[s_next] if s_next else 0))
            v_old = V[s]
            V[s] = max(q_vals) if q_vals else v_old
            delta = max(delta, abs(V[s] - v_old))
        if delta < tol:
            break

    # extrae política óptima
    pi = {}
    for s in estados:
        if s is None:
            pi[s] = []
        else:
            best_q = -float('inf')
            choices = []
            for a, (s_next, r) in transiciones[s].items():
                q = r + gamma * (V[s_next] if s_next else 0)
                if q > best_q:
                    best_q = q
                    choices = [a]
                elif q == best_q:
                    choices.append(a)
            pi[s] = choices
    return V, pi
   
# imprime valores y política en tablas
def imprimir_tablas(gamma, V, pi):
    print(f"\n=== Valores óptimos V* (gamma = {gamma}) ===")
    header = "Nodo\\Dinero | " + " ".join(f"{d:>6}" for d in range(11))
    print(header)
    print("-" * len(header))
    for nodo in range(4):
        row = f"{nodo:>10} | " + " ".join(f"{V[(nodo,d)]:6.2f}" for d in range(11))
        print(row)
    print(f"\n=== Política óptima π* (gamma = {gamma}) ===")
    header = "Nodo\\Dinero | " + " ".join(f"{d:>8}" for d in range(11))
    print(header)
    print("-" * len(header))
    for nodo in range(4):
        row = f"{nodo:>10} | " + " ".join(f"{','.join(map(str,pi[(nodo,d)])) or '-':>8}" for d in range(11))
        print(row)

# Iteración de valor para distintos gamma
for gamma in [0.99,0.1,1]:
    V_opt, pi_opt = value_iteration(gamma)
    imprimir_tablas(gamma, V_opt, pi_opt)




=== Valores óptimos V* (gamma = 0.99) ===
Nodo\Dinero |      0      1      2      3      4      5      6      7      8      9     10
------------------------------------------------------------------------------------------
         0 |  -1.00   0.00   0.00   0.87   1.83   1.94   3.00   3.00   3.87   4.83   4.98
         1 | -2000.00   0.00   0.00   0.86   1.81   2.00   2.97   2.97   3.83   4.78   4.93
         2 | -2000.00   0.00   0.00   0.86   1.81   2.00   2.97   2.97   3.83   4.78   4.93
         3 |   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00   0.00

=== Política óptima π* (gamma = 0.99) ===
Nodo\Dinero |        0        1        2        3        4        5        6        7        8        9       10
----------------------------------------------------------------------------------------------------------------
         0 |        2        1        1        1        1        2        1        1        1        1        1
         1 |      0,3        0

 γ = 0.1 ⇒ la política casi ignora el futuro: prioriza evitar castigos inmediatos.  
 γ = 1   ⇒ valora premios lejanos, por eso arriesga comprando/vendiendo antes del nodo 3.  
 γ = 0.99 es el intermedio del enunciado: balancea riesgo-recompensa y evita quedar sin $$.